# Vectorless RAG: Multi-Hop Claims Adjudication & Benefit Tables

This notebook demonstrates two advanced use cases where Vectorless RAG outperforms standard chunking-based RAG:

1. **Multi-Hop Claims Adjudication** — fetching discrete variables (dates, deductibles, limits) from different parts of a policy without introducing noise.
2. **Benefit Tables & Rate Charts** — retrieving whole logical nodes that preserve table structure, headers, and footnotes.

Standard Vector RAG often fails here because:
- Chunking shatters tables and cross-references.
- Embedding similarity retrieves irrelevant nearby text.
- Multi-hop reasoning requires fetching multiple distant nodes, which chunking mixes together.

## 1. Prerequisites

This notebook assumes you have:

- A complex insurance policy PDF in a local `data/` folder.
- A PageIndex API key for tree generation.
- A Groq API key for the free LLM endpoint.

The LLM layer uses [Groq](https://groq.com/) via its OpenAI-compatible endpoint. You can choose between `llama-3.3-70b-versatile` (larger) and `llama-3.1-8b-instant` (faster) at runtime.

### API Key Setup

You will need two API keys:

| Key | Where to get it |
|-----|------------------|
| `GROQ_API_KEY` | [https://console.groq.com/keys](https://console.groq.com/keys) |
| `PAGEINDEX_API_KEY` | [https://www.pageindex.ai](https://www.pageindex.ai) |

Set them as environment variables or in a `.env` file in the same directory as this notebook:

```
GROQ_API_KEY=gsk_...
PAGEINDEX_API_KEY=...
```

In [ ]:
%pip install -q --upgrade pageindex openai python-dotenv pymupdf

## 2. Configuration

The notebook reads environment variables from a `.env` file or your shell:

- `GROQ_API_KEY`: Groq API key (from [console.groq.com/keys](https://console.groq.com/keys)).
- `PAGEINDEX_API_KEY`: PageIndex API key.
- `PDF_NAME`: Optional filename in `data/` if you want to pin a specific PDF.

You will be prompted to choose between two Groq models at runtime:
- `llama-3.3-70b-versatile` — larger, more capable.
- `llama-3.1-8b-instant` — faster, lighter.

Groq provides a free OpenAI-compatible endpoint, so no `LLM_BASE_URL` is needed.

In [ ]:
from pathlib import Path
import json
import os
import re
import time
from typing import Any

from dotenv import load_dotenv
from openai import AsyncOpenAI
from pageindex import PageIndexClient
import pageindex.utils as pi_utils

load_dotenv()

DATA_DIR = Path("data")
CACHE_DIR = DATA_DIR / "cache"
PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY", "").strip()
LLM_API_KEY = os.getenv("GROQ_API_KEY", "").strip()
LLM_BASE_URL = "https://api.groq.com/openai/v1"
PDF_NAME = os.getenv("PDF_NAME")

MODELS = {
    "1": ("llama-3.3-70b-versatile", "Llama 3.3 70B — larger, more capable"),
    "2": ("llama-3.1-8b-instant", "Llama 3.1 8B — faster, lighter"),
}

print("Choose an LLM model:")
for key, (name, desc) in MODELS.items():
    print(f"  {key}. {name} — {desc}")

choice = input("Enter 1 or 2: ").strip()
LLM_MODEL = MODELS.get(choice, MODELS["1"])[0]
print(f"Using model: {LLM_MODEL}")

if not PAGEINDEX_API_KEY:
    print("Set PAGEINDEX_API_KEY before running the PageIndex cells.")
if not LLM_API_KEY:
    print("Set GROQ_API_KEY before running the retrieval and answer cells.")

pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY) if PAGEINDEX_API_KEY else None
llm_client = AsyncOpenAI(api_key=LLM_API_KEY, base_url=LLM_BASE_URL) if LLM_API_KEY else None


def extract_json(text: str) -> dict[str, Any]:
    match = re.search(r"\{.*\}", text, re.S)
    if not match:
        raise ValueError(f"Expected JSON output, got: {text[:500]}")
    return json.loads(match.group(0))


async def call_llm(system_prompt: str, user_prompt: str, model: str = LLM_MODEL, temperature: float = 0.0) -> str:
    if llm_client is None:
        raise RuntimeError("GROQ_API_KEY is not configured.")
    response = await llm_client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content.strip()


def preview_text(text: str, limit: int = 1200) -> str:
    text = text.strip()
    return text if len(text) <= limit else text[:limit].rstrip() + "..."

## 3. Load a PDF from `data/`

This notebook works with one user-provided PDF at a time. If `PDF_NAME` is set, the notebook uses that file from `data/`; otherwise it uses the first PDF it finds.

For this lab, use a complex insurance policy PDF that contains multi-section clauses, effective dates, deductibles, and rate tables.

In [ ]:
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Missing data folder: {DATA_DIR.resolve()}")

pdf_candidates = sorted(DATA_DIR.glob("*.pdf"))
if not pdf_candidates:
    raise FileNotFoundError(f"No PDF files found in {DATA_DIR.resolve()}")

if PDF_NAME:
    matching = [path for path in pdf_candidates if path.name == PDF_NAME]
    if not matching:
        available = ", ".join(path.name for path in pdf_candidates)
        raise FileNotFoundError(f"PDF_NAME={PDF_NAME!r} was not found in data/. Available files: {available}")
    pdf_path = matching[0]
else:
    pdf_path = pdf_candidates[0]

print(f"Using PDF: {pdf_path}")

## 4. Build the PageIndex tree

This cell submits the selected PDF to PageIndex and retrieves the generated tree.

The tree is cached locally in `data/cache/` so subsequent runs skip the PageIndex API call for the same document.

If the document is still processing, the notebook polls with a progress bar until the tree is ready.

In [ ]:
CACHE_DIR.mkdir(exist_ok=True)
cache_path = CACHE_DIR / f"{pdf_path.stem}_tree.json"

if cache_path.exists():
    print(f"Loading cached tree from {cache_path}")
    tree = json.loads(cache_path.read_text())
else:
    if pi_client is None:
        raise RuntimeError("PAGEINDEX_API_KEY is not configured.")

    submitted = pi_client.submit_document(str(pdf_path))
    doc_id = submitted.get("doc_id") or submitted.get("result", {}).get("doc_id")
    if not doc_id:
        raise RuntimeError(f"Could not read doc_id from PageIndex response: {submitted}")

    print(f"Submitted document id: {doc_id}")
    print("Waiting for PageIndex to process the document...")

    poll_interval = 5
    max_wait = 300
    elapsed = 0
    spinner = ["|", "/", "-", "\\"]
    while elapsed < max_wait:
        if pi_client.is_retrieval_ready(doc_id):
            break
        idx = (elapsed // poll_interval) % len(spinner)
        print(f"\r  {spinner[idx]}  Waiting... {elapsed}s / {max_wait}s", end="", flush=True)
        time.sleep(poll_interval)
        elapsed += poll_interval
    else:
        raise RuntimeError(
            f"PageIndex did not finish within {max_wait}s. Rerun this cell later."
        )
    print(f"\r  Done! Processed in {elapsed}s.{' ' * 20}")

    tree_response = pi_client.get_tree(doc_id, node_summary=True)
    tree = tree_response.get("result", tree_response)

    cache_path.write_text(json.dumps(tree))
    print(f"Tree cached to {cache_path}")

print("Tree ready. Top-level preview:")
pi_utils.print_tree(tree[:2] if isinstance(tree, list) else tree)

## 5. Prepare retrieval helpers

The next cells use the tree structure for two passes:

1. Tree search: the LLM selects the node IDs most likely to contain the answer.
2. Evidence extraction: the notebook pulls the full text for those nodes and passes it into the final answer prompt.

This keeps the workflow vectorless while still giving you transparent, inspectable retrieval decisions.

In [ ]:
node_map = pi_utils.create_node_mapping(tree)

def tree_as_prompt_text(document_tree: object) -> str:
    tree_copy = document_tree.copy() if isinstance(document_tree, dict) else document_tree
    tree_without_text = pi_utils.remove_fields(tree_copy, fields=["text"])
    return json.dumps(tree_without_text, indent=2)


def collect_node_text(node_ids: list[str]) -> str:
    parts: list[str] = []
    for node_id in node_ids:
        node = node_map.get(node_id)
        if not node:
            continue
        text = node.get("text", "")
        title = node.get("title", node_id)
        page_index = node.get("page_index", "?")
        parts.append(f"[node={node_id} page={page_index} title={title}]\n{text}")
    return "\n\n".join(parts)


print(f"Indexed nodes: {len(node_map)}")

---

## Scenario A: Multi-Hop Claims Adjudication

### The Problem

When adjudicating an insurance claim, you often need to cross-reference **multiple distant sections** of a policy:

- **Effective Dates** (when does coverage apply?)
- **Deductible Limits** (what does the insured pay first?)
- **Coverage Caps** (what is the maximum payout?)

Standard chunking-based RAG retrieves a single chunk. If the effective dates and deductible limits are in different sections (pages 2 and 15), chunking either misses one or mixes in irrelevant surrounding text.

Vectorless RAG solves this by letting the LLM reason over the **entire tree** and explicitly select multiple node IDs — even if they are far apart in the document.

### Step A1: Define the adjudication question

Ask a question that requires fetching facts from multiple sections of the policy.

In [ ]:
ADJUDICATION_QUESTION = input(
    "Enter a multi-hop claims question (e.g., 'Is this claim within the effective dates and below the deductible limit?'): "
).strip()
if not ADJUDICATION_QUESTION:
    raise ValueError("A question is required to continue.")

### Step A2: Multi-hop tree search

The LLM receives the full document tree and explicitly identifies **which sections** to fetch. It returns a structured response with:
- `thinking`: why each node was selected.
- `node_list`: the node IDs to retrieve.

This is the key advantage over chunking: the LLM can select **any combination of nodes** across the entire document.

In [ ]:
retrieval_system_prompt = """
You are a document retrieval assistant for a multi-hop insurance claims adjudication system.

You will receive a user question and a PageIndex tree made of node titles and summaries.
The question may require information from MULTIPLE distant sections of the policy.

Identify ALL node IDs that contain relevant evidence. Common patterns:
- Effective Dates / Policy Period
- Deductible Limits / Waiting Periods
- Coverage Caps / Sublimits
- Exclusions / Conditions
- Definition sections that clarify terms used elsewhere

Return valid JSON with this shape:
{
  "thinking": "step-by-step reasoning about which sections are needed and why",
  "node_list": ["node_id_1", "node_id_2", ...]
}

Do not output markdown, prose, or extra keys.
Be thorough — missing a required node means an incomplete adjudication.
""".strip()

retrieval_user_prompt = f"""
Question:
{ADJUDICATION_QUESTION}

PageIndex tree:
{tree_as_prompt_text(tree)}
""".strip()

retrieval_response = await call_llm(retrieval_system_prompt, retrieval_user_prompt)
retrieval_json = extract_json(retrieval_response)
selected_node_ids = retrieval_json.get("node_list", [])

print(f"Selected {len(selected_node_ids)} node(s):", selected_node_ids)
print("\nRetrieval reasoning:")
print(retrieval_json.get("thinking", ""))

### Step A3: Extract multi-hop evidence

Each selected node is retrieved in full. Because Vectorless RAG fetches **whole logical nodes**, there is no chunking noise — just clean, relevant passages from different parts of the policy.

In [ ]:
evidence_text = collect_node_text(selected_node_ids)
if not evidence_text.strip():
    raise RuntimeError("No evidence text was collected. Check the selected node IDs and tree mapping.")

print("Evidence preview (multi-hop):\n")
print(preview_text(evidence_text, limit=4000))

### Step A4: Generate adjudication answer

The final LLM call receives all the multi-hop evidence and produces a structured adjudication response.

In [ ]:
adjudication_system_prompt = """
You are an insurance claims adjudication assistant.

You have been given evidence from multiple sections of an insurance policy.
Analyze the evidence to answer the adjudication question.

Return valid JSON with this shape:
{
  "decision": "approve" or "deny" or "needs_review",
  "final_answer": "clear, concise answer for the claims adjuster",
  "explainability": {
    "effective_dates": "policy period information found",
    "deductible": "deductible information found",
    "coverage_limits": "coverage cap information found",
    "evidence_used": ["short evidence note 1", "short evidence note 2"]
  }
}

Ground every statement in the provided evidence. Do not invent facts.
""".strip()

adjudication_user_prompt = f"""
Adjudication question:
{ADJUDICATION_QUESTION}

Evidence from policy:
{evidence_text}
""".strip()

adjudication_response = await call_llm(adjudication_system_prompt, adjudication_user_prompt)
adjudication_json = extract_json(adjudication_response)

print("Decision:", adjudication_json.get("decision", ""))
print("\nFinal answer:")
print(adjudication_json.get("final_answer", ""))
print("\nExplainability:")
print(json.dumps(adjudication_json.get("explainability", {}), indent=2))

---

## Scenario B: Benefit Tables & Rate Charts

### The Problem

Insurance policies contain structured tables — rate charts, benefit schedules, co-pay matrices — that are critical for accurate answers.

Standard chunking **shatters** tables:
- A table spanning one page might be split into 3-4 chunks.
- Headers get separated from their rows.
- Footnotes referencing table cells get lost.
- The LLM receives disconnected table fragments with no structural context.

Vectorless RAG retrieves **whole logical nodes**. PageIndex preserves table structure as a single node, so the LLM receives the complete table with headers, data, and footnotes intact.

### Step B1: Define the table query

Ask a question that requires reading a specific rate chart or benefit table.

In [ ]:
TABLE_QUESTION = input(
    "Enter a table-related question (e.g., 'What is the coinsurance rate for out-of-network providers?'): "
).strip()
if not TABLE_QUESTION:
    raise ValueError("A question is required to continue.")

### Step B2: Targeted tree search for tables

The LLM identifies which node(s) contain the relevant table. Because PageIndex preserves table structure, the LLM can reason about table headers and row labels in the tree.

In [ ]:
table_retrieval_prompt = """
You are a document retrieval assistant specializing in structured data.

You will receive a user question and a PageIndex tree made of node titles and summaries.
The question likely requires information from a TABLE, CHART, or SCHEDULE in the document.

Look for nodes that contain:
- Rate charts / premium tables
- Benefit schedules / co-pay matrices
- Coverage limit tables
- Any node whose title or summary suggests tabular data

Return valid JSON with this shape:
{
  "thinking": "which table or schedule was identified and why it matches the question",
  "node_list": ["node_id_1", "node_id_2"]
}

Do not output markdown, prose, or extra keys.
Prefer retrieving the ENTIRE table node over partial fragments.
""".strip()

table_user_prompt = f"""
Question:
{TABLE_QUESTION}

PageIndex tree:
{tree_as_prompt_text(tree)}
""".strip()

table_response = await call_llm(table_retrieval_prompt, table_user_prompt)
table_json = extract_json(table_response)
table_node_ids = table_json.get("node_list", [])

print(f"Selected {len(table_node_ids)} node(s):", table_node_ids)
print("\nRetrieval reasoning:")
print(table_json.get("thinking", ""))

### Step B3: Extract table evidence

The full table node is retrieved. Notice how the complete structure — headers, rows, footnotes — is preserved.

In [ ]:
table_evidence = collect_node_text(table_node_ids)
if not table_evidence.strip():
    raise RuntimeError("No table evidence was collected. Check the selected node IDs.")

print("Table evidence preview:\n")
print(preview_text(table_evidence, limit=4000))

### Step B4: Generate table-based answer

The final LLM call reads the complete table and answers the specific rate/benefit question.

In [ ]:
table_answer_prompt = """
You are an insurance benefits assistant answering questions from rate charts and tables.

You have been given complete table data extracted from a policy document.
Answer the user's question by reading the table carefully.

Return valid JSON with this shape:
{
  "final_answer": "specific answer with exact numbers, rates, or limits from the table",
  "table_structure": {
    "table_title": "name or section of the table",
    "relevant_row": "the specific row or cell that answers the question",
    "relevant_column": "the specific column or category"
  },
  "explainability": {
    "evidence_used": ["specific table excerpt that was used"]
  }
}

Always include exact numbers and rates from the table. Do not approximate.
""".strip()

table_answer_user = f"""
Question:
{TABLE_QUESTION}

Table data:
{table_evidence}
""".strip()

table_final_response = await call_llm(table_answer_prompt, table_answer_user)
table_final_json = extract_json(table_final_response)

print("Final answer:")
print(table_final_json.get("final_answer", ""))
print("\nTable reference:")
print(json.dumps(table_final_json.get("table_structure", {}), indent=2))
print("\nExplainability:")
print(json.dumps(table_final_json.get("explainability", {}), indent=2))